# Lab 06 — Analyzing an Information Extraction Pilot

In this notebook, we examine the results of an information-extraction pilot conducted in Label Studio. Unlike classification, information extraction identifies specific information within a document, such as a text value or a region in an image.

We will explore annotation progress, effort, extracted information, differences from reference data, and potential problems with the task design. Formal agreement and annotation-quality methods will be covered in Lab 07.

### Load the pilot

The CORD example selects the OCR region containing a receipt's final total. Set `REFERENCE_FILE` or `DOCUMENTS_FILE` to `None` if they are not available. For text fields, see the examples in `utils/ie_analysis.py`.

In [ ]:
import pandas as pd
from utils.ie_analysis import load_ie_pilot

ANNOTATIONS_FILE = "cord-results.json"
REFERENCE_FILE = "export/cord_reference.csv"
DOCUMENTS_FILE = "../../project_baselines/project_04/data/prepared/cord_v2/documents.jsonl"

items, annotations, extractions = load_ie_pilot(
    ANNOTATIONS_FILE, REFERENCE_FILE, documents_file=DOCUMENTS_FILE
)

## 1. What happened in our pilot?

How many documents were annotated, and did they receive the requested number of submissions? A submitted annotation can still contain no usable response: for example, an annotator may submit the preloaded OCR boxes without selecting a total.

In [ ]:
from IPython.display import display
from utils.analysis import pilot_summary, plot_annotation_coverage

NUM_RATINGS_PER_EXAMPLE = 2

display(pilot_summary(items, annotations, NUM_RATINGS_PER_EXAMPLE))
display(annotations["response_status"].value_counts().rename("submitted responses"))
fig, ax = plot_annotation_coverage(items, NUM_RATINGS_PER_EXAMPLE)

## 2. Do annotators improve over time?

Does the time needed to identify the requested information change as annotators gain experience? We use the recorded submission order for each worker. Faster responses do not by themselves establish better extraction quality; difficulty, interruptions, and fatigue can also affect time.

In [ ]:
from utils.analysis import plot_annotator_progress

fig, ax = plot_annotator_progress(annotations)

## 3. Where might there be problems?

### 3.1 Annotation time

Some documents take longer than others. This may indicate a difficult decision or missing information, but a long time can also reflect an interruption. What do the slowest examples have in common?

In [ ]:
from utils.analysis import plot_annotation_times

plot_annotation_times(annotations)
annotations.sort_values("lead_time", ascending=False)[
    ["example_id", "annotator_id", "response_status", "lead_time"]
].head(5)

### 3.2 Differences from reference information

A difference may reflect an unclear definition, an ambiguous receipt, OCR problems, or an incomplete reference. An empty reference line list does not establish that a total is absent. Inspect some differences and cases without a comparable reference.

In [ ]:
differences = extractions[extractions["comparison"] != "match"]
differences[[
    "example_id", "field_name", "region_id", "value",
    "reference_region_ids", "reference_value", "comparison"
]].head(10)

### 3.3 Are some fields more difficult than others?

Compare response outcomes by field, keeping the counts in view. With only one field, this is a summary of outcomes rather than a comparison of difficulty. Annotation time belongs to the whole task, not to an individual field.

In [ ]:
outcomes = pd.crosstab(extractions["field_name"], extractions["comparison"])
display(outcomes)

In [ ]:
extractions.groupby(["field_name", "status"]).size().rename("responses")

### 3.4 Does document complexity matter?

A receipt with more OCR lines, or a longer text document, may take more effort to inspect. Complexity alone does not determine difficulty. What pattern do you observe?

In [ ]:
effort = annotations.merge(items[["example_id", "text", "ocr_line_count"]], on="example_id", how="left")
if effort["ocr_line_count"].notna().any():
    effort["complexity"] = effort["ocr_line_count"]
    complexity_label = "Available OCR lines"
else:
    effort["complexity"] = effort["text"].str.len()
    complexity_label = "Text length (characters)"

effort.plot.scatter(x="complexity", y="lead_time", figsize=(7, 4),
                    xlabel=complexity_label, ylabel="Annotation time (seconds)")

## 4. Take a closer look

Choose an interesting document and inspect its information and individual responses. Do the source, the OCR, the reference, or the instructions help explain what happened? Change `EXAMPLE_ID` to explore another document.

In [ ]:
from utils.ie_analysis import show_images

EXAMPLE_ID = "cord_train_0256"

# Display the source image, when available
if "image" in items.columns and items["image"].notna().any():
    show_images(items, EXAMPLE_ID)

# Inspect the extracted information
details = extractions[extractions["example_id"] == EXAMPLE_ID]

display(details[
    ["annotator_id", "field_name", "region_id",
     "value", "reference_region_ids", "comparison"]
])

## 5. Estimating annotation costs

Label Studio records time for each submitted annotation. These estimates may include interruptions and exclude preparation, training, and review. A small pilot may not represent a larger task.

### 5.1 How much did our pilot cost?

Estimate the cost from recorded submission times and a hypothetical hourly rate. Count each submission once, even when it contains several fields.

In [ ]:
HOURLY_RATE = 18  # EUR per hour — hypothetical rate

costs = annotations[annotations["status"] == "submitted"].dropna(subset=["lead_time"]).copy()
costs = costs[costs["time_issue"] == ""]
costs["cost"] = costs["lead_time"] / 3600 * HOURLY_RATE

print(f"Total annotation time: {costs['lead_time'].sum() / 60:.1f} minutes")
print(f"Estimated pilot cost: €{costs['cost'].sum():.2f}")
print(f"Average cost per annotation: €{costs['cost'].mean():.2f}")

### 5.2 Scaling up

How would the estimate change with more documents or three independent annotations per document? How reliable is it for a small pilot?

In [ ]:
NUM_EXAMPLES = 1000
RATINGS_PER_EXAMPLE = 2

estimated_cost = NUM_EXAMPLES * RATINGS_PER_EXAMPLE * costs["cost"].mean()
print(f"Estimated annotation cost: €{estimated_cost:,.2f}")

**Discussion**

- How much would it cost to annotate your full dataset?
- What changes if each document requires three annotations?
- How could effort be reduced without compromising usefulness?

## 6. What should we improve?

Identify two or three observations worth investigating. Consider unclear field definitions, ambiguous or missing source information, OCR errors or missing regions, incomplete references, interface problems, and the effort needed to find or enter values.

For each observation, propose a possible cause and a revision. If annotators gave qualitative feedback, inspect it as well. Which changes would you test in a second pilot?

Can we conclude from this pilot that annotations are reliable? We will explore this question in Lab 07.

In [ ]:
# If your task collected qualitative feedback, inspect it here.
# display(annotations[["example_id", "feedback_text"]].dropna())
# display(annotations["feedback_category"].value_counts())